# 03 예측 백테스팅 — 감 · 요일 평균 · TimesFM (S15P31A308-57)

**결론**: (샘플) 위글리 배달 이력 수령 후 작성. 샘플 데이터 결과는 계산 틀 확인용이고 채택 판정에 쓰지 않는다.

- 평가 지표 정의 4절: 마지막 8주, 기준일마다 그날까지의 데이터로 h = 1~7 예측, 기준 지표 h = 1
- 채택: 채널 합계 · h = 1 · 같은 행에서 TimesFM WAPE < 요일 평균 WAPE, Bias ±10% 는 별도 합격 조건
- TimesFM 은 `uv sync --group batch` 후 `USE_TIMESFM = True` (CPU 로 8주 약 8분)
- 실데이터는 `DATA_DIR` 만 바꿔 다시 돌린다. 같은 표를 `scripts/backtest_forecast.py` 가 md 로 쓴다

In [ ]:
from pathlib import Path

import pandas as pd

from weglee_ai.backtest import ALL_CHANNELS, LastWeekPredictor, backtest, eval_period, score, verdict, with_channel_sum
from weglee_ai.baseline import DowAvgPredictor
from weglee_ai.metrics import wape

DATA_DIR = Path("..") / "data" / "sample"  # 위글리 실데이터는 ../data/processed
EVAL_WEEKS = 8
USE_TIMESFM = False
pd.set_option("display.float_format", "{:,.3f}".format)

In [ ]:
sales = pd.read_csv(DATA_DIR / "pos_sales.csv", parse_dates=["sold_date"])
predictors = [LastWeekPredictor(), DowAvgPredictor()]
if USE_TIMESFM:
    from weglee_ai.batch.predictors import TimesFmPredictor

    predictors.append(TimesFmPredictor())

rows = backtest(sales, predictors, eval_weeks=EVAL_WEEKS)
first, last = eval_period(sales, EVAL_WEEKS)
print(f"평가 기간 {first} ~ {last} · 행 {len(rows):,}")
print("결론:", verdict(rows))

## 발표용 — 채널 합계 · h = 1

In [ ]:
table = score(rows)
table[(table["구간"] == "h=1") & (table["채널"] == ALL_CHANNELS)][["모델", "WAPE", "Bias", "행"]]

## 전체 — 모델 × 채널 × 구간

In [ ]:
table

## h 별 WAPE (채널 합계)

멀리 볼수록 얼마나 나빠지는지. 리드타임 · 발주 주기 동안의 소비 예측에 쓰인다.

In [ ]:
summed = with_channel_sum(rows)
summed = summed[summed["channel"] == ALL_CHANNELS]
summed.groupby(["h", "model"]).apply(lambda g: wape(g.actual, g.predicted), include_groups=False).unstack("model")